<a href="https://colab.research.google.com/github/ayesha112244/handwashing-stage-classification-cnn/blob/main/label_error_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook focuses exclusively on identifying and removing potential label errors using data-centric AI (Confident Learning via Cleanlab), after which a cleaned dataset is saved for further preprocessing and model training.

## 1. Install & Import Libraries

In [ ]:
!pip install cleanlab

In [ ]:
import os
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import StratifiedKFold
import cleanlab

In [ ]:
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

Fixing random seeds ensures reproducible results and consistent label error estimation.

## 2. Loading Dataset

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
dataset_path = "/content/drive/MyDrive/CHS2406_Coursework2_Dataset_2526/CHS2406_Coursework2_Data_Repository"

images = []
labels = []
image_size = 150

for label, stage in enumerate(sorted(os.listdir(dataset_path))):
    stage_path = os.path.join(dataset_path, stage)

    for img_name in os.listdir(stage_path):
        img_path = os.path.join(stage_path, img_name)

        img = cv2.imread(img_path)
        if img is None:
            continue

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (image_size, image_size))

        images.append(img)
        labels.append(label)

images = np.array(images)
labels = np.array(labels)

print("Total images loaded:", images.shape[0])
print("Image shape:", images.shape)
print("Total labels:", labels.shape[0])


Total images: 8538


Within this step, the data was retrieved from the Google Drive by traversing the folders, considering that each folder corresponds to a precise step within the hand washing process. All the pictures were loaded from the Google Drive using the OpenCV library and were normalized to have the same size of 150x150 pixels. A validation step was included to ignore the unreadable files that were neither readable nor supported. Each corresponding stage received an integer label, and the data was stored within NumPy arrays.

In [ ]:
# Normalization
X = images.astype("float32") / 255.0
y = labels

## 3. Data Labelling Errors

### 3.1 Understanding Label Errors

Label errors occur when an image is assigned to an incorrect class label that does not match its visual content. In this dataset, label errors mainly arise due to manual data collection and the visual similarity between consecutive handwashing stages.

Since several stages share similar hand positions and movements, some images may have been placed into incorrect folders. Such incorrect labels can negatively affect model training by confusing the learning process.

Therefore, identifying and correcting label errors is an essential step to improve dataset reliability and overall model performance.


## 3.2. Detecting Label Errors using Confident Learning (Cleanlab)

### 3.2.1 Method Overview
To identify potential label errors in a dataset, a **data-centric AI methodology** employing **Confident Learning** was implemented utilizing the Cleanlab library. This approach minimizes the need for manual inspection of large volumes of images by pinpointing instances where a trained model confidently predicts a label different from the provided one.

A lightweight **Convolutional Neural Network (CNN)** was developed and evaluated through out-of-sample predictions using Stratified K-Fold cross-validation, ensuring that each label assessment stemmed from a model that had not previously encountered the specific image, thus preventing biased or overly optimistic evaluations.

This technique is noted for its computational efficiency, suitability for academic purposes, and effectiveness in large-scale dataset quality analysis.

In [ ]:

def build_model():
    model = Sequential([
        Conv2D(16, (3,3), activation='relu', input_shape=(150,150,3)),
        MaxPooling2D(2,2),

        Conv2D(32, (3,3), activation='relu'),
        MaxPooling2D(2,2),

        Flatten(),
        Dense(64, activation='relu'),
        Dense(8, activation='softmax')
    ])

    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

In [ ]:
#OUT-OF-SAMPLE Predictions
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

pred_probs = np.zeros((len(X), 8))

for train_idx, val_idx in skf.split(X, y):
    model = build_model()
    model.fit(
        X[train_idx],
        to_categorical(y[train_idx], 8),
        epochs=3,
        batch_size=32,
        verbose=0
    )
    pred_probs[val_idx] = model.predict(X[val_idx], verbose=0)


During this step, the images underwent normalization before being used as input data for training the simple CNN model. The simple CNN model was not intended for classification but rather for predicting class probability distributions for each image.

Stratified K-Fold cross-validation was used for creating predictions on which the probabilities for errors in labelling could be calculated accurately.

### 3.2.3 Detected Label Errors per Stage

In [ ]:
#Detect Label Errors using CLEANLAB
label_error_indices = cleanlab.filter.find_label_issues(
    labels=y,
    pred_probs=pred_probs,
    return_indices_ranked_by="self_confidence"
)

# Keep only top 20% most confident label errors
num_to_remove = int(0.2 * len(label_error_indices))
label_error_indices = label_error_indices[:num_to_remove]


Using Cleanlab’s confident learning approach, potential label inconsistencies were identified based on model self-confidence scores. Images for which the model strongly disagreed with the given label were flagged as potential label errors.

Most detected errors occurred between visually similar handwashing stages, where distinguishing features are subtle. To avoid excessive data removal, only the top-ranked most confident label errors were selected for removal.


## 3.3. Count Label Errors Per Stage

In [ ]:
#Count Label Errors Per Stage
error_counts = {}

for stage in range(8):
    error_counts[f"Stage {stage+1}"] = np.sum(y[label_error_indices] == stage)

for k, v in error_counts.items():
    print(f"{k}: {v}")


Stage 1: 121
Stage 2: 104
Stage 3: 94
Stage 4: 98
Stage 5: 89
Stage 6: 124
Stage 7: 146
Stage 8: 78


## 3.4 Dataset Size After Cleaning

In [ ]:
#REMOVE Label Errors (Cleaning)
mask = np.ones(len(X), dtype=bool)
mask[label_error_indices] = False

clean_images = images[mask]
clean_labels = labels[mask]

print("Original size:", len(images))
print("Cleaned size:", len(clean_images))


Original size: 8538
Cleaned size: 7684


- Original dataset size: 8,538 images  
- Cleaned dataset size: 7,684 images  

The reduction reflects the removal of samples that were most confidently identified as mislabeled, resulting in a more reliable dataset for subsequent preprocessing and model training.


In [ ]:
#saving clean dataset
np.save("/content/drive/MyDrive/CHS2406_Coursework2_Dataset_2526/cleaned_dataset/clean_images.npy", clean_images)
np.save("/content/drive/MyDrive/CHS2406_Coursework2_Dataset_2526/cleaned_dataset/clean_labels.npy", clean_labels)
